# Online Food Dataset — Exploratory Data Analysis & Cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/Online_Food_Dataset_EDA_and_Cleaning.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, online yemek siparişi veren müşterilerin demografik özellikleri, gelir seviyeleri ve verdikleri geri bildirimleri içeren **Online Food Dataset** üzerinde kapsamlı bir **Keşifsel Veri Analizi (EDA)** ve **Veri Temizliği (Data Cleaning)** süreci uygulanmıştır. Yinelenen (duplicate) veriler, gereksiz sütunlar (`Unnamed: 12`), kategorik metinlerdeki boşluk hataları temizlenmiş ve demografik dağılımlar görselleştirilmiştir.

### 1. Veri Setinin İndirilmesi / Download Dataset

In [ ]:
import kagglehub
import os

# Kaggle üzerinden online food veri setini indirelim
path = kagglehub.dataset_download("sudarshan24byte/online-food-dataset")
print("Path to dataset files:", path)

### 2. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Görselleştirme teması
sns.set_theme(style="whitegrid", palette="muted")
print("Kütüphaneler hazır!")

### 3. Verinin Yüklenmesi ve İlk Bakış / Load Data & Initial Inspection

In [ ]:
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Veri Boyutu: {df.shape[0]} satır, {df.shape[1]} sütun\n")
display(df.head())

### 4. Veri Tipleri, Eksik Değer ve Özet İstatistikler / Data Info & Summary

In [ ]:
print("--- Sütun Bilgileri ve Eksik Değerler ---")
df.info()

print("\n--- Eksik Değer Sayıları ---")
print(df.isnull().sum())

print("\n--- Sayısal ve Kategorik Sütunların Özeti ---")
display(df.describe(include='all'))

### 5. Yinelenen (Duplicate) Satır ve Sütun Analizi / Redundancy Analysis

In [ ]:
# 1. 'Output' ve 'Unnamed: 12' sütunlarının benzerlik kontrolü
if 'Unnamed: 12' in df.columns:
    same_values = (df['Output'] == df['Unnamed: 12']).sum()
    print(f"Output ve Unnamed: 12 sütunlarının tamamen aynı olduğu satır sayısı: {same_values} / {len(df)}")

# 2. Tekrarlanan (Duplicate) satır kontrolü
duplicate_count = df.duplicated().sum()
print(f"Yinelenen (Duplicate) satır sayısı: {duplicate_count}")

if duplicate_count > 0:
    print("\nYinelenen satırlardan ilk 5 örnek:")
    display(df[df.duplicated()].head())

### 6. Kategorik Sütunların Benzersiz Değerleri / Categorical Analysis

In [ ]:
categorical_cols = ['Gender', 'Marital Status', 'Occupation', 'Monthly Income', 'Educational Qualifications', 'Feedback']
print("--- Kategorik Sütunların Benzersiz Değer Dağılımları ---")
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col} sütunu benzersiz değerleri:")
        print(df[col].value_counts())

### 7. Veri Temizliği / Data Cleaning Pipeline

In [ ]:
print(f"Temizlik öncesi satır sayısı: {df.shape[0]}")

# 1. Yinelenen satırları kaldırıyoruz
df_cleaned = df.drop_duplicates().copy()

# 2. Gereksiz ve kopya olan 'Unnamed: 12' sütununu kaldırıyoruz
if 'Unnamed: 12' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 12'])

# 3. Kategorik sütunların başındaki ve sonundaki olası boşlukları temizleyelim
for col in df_cleaned.select_dtypes(include=['object']).columns:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip()

print(f"Temizlik sonrası satır sayısı: {df_cleaned.shape[0]}")
print(f"Silinen tekrarlayan satır sayısı: {df.shape[0] - df_cleaned.shape[0]}")

print("\nTemizlenmiş Veri Setinin İlk 5 Satırı:")
display(df_cleaned.head())

### 8. Keşifsel Veri Görselleştirmesi / Exploratory Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Geri Bildirim Dağılımı
sns.countplot(data=df_cleaned, x='Feedback', ax=axes[0], palette='pastel')
axes[0].set_title('Müşteri Geri Bildirim Dağılımı (Pozitif / Negatif)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Geri Bildirim')
axes[0].set_ylabel('Müşteri Sayısı')

# Mesleğe Göre Geri Bildirim Dağılımı
sns.countplot(data=df_cleaned, x='Occupation', hue='Feedback', ax=axes[1], palette='Set2')
axes[1].set_title('Mesleklere Göre Geri Bildirim Dağılımı', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Meslek')
axes[1].set_ylabel('Kişi Sayısı')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### 9. Sonuç / Conclusion

- Veri setindeki yinelenen satırlar ve fazlalık 'Unnamed: 12' sütunu temizlenmiştir.
- Kategorik metinlerdeki boşluk hataları giderilerek makine öğrenmesi algoritmaları için temiz bir veri tabanı oluşturulmuştur.
- Analiz sonuçlarına göre müşteri geri bildirimlerinin büyük bir çoğunluğu pozitiftir ve öğrenci ile çalışan grupları siparişlerin en yoğun olduğu segmentlerdir.